# Crop × Mars: exploratory similarity pipeline
Project author: Lissandra Kruse Fuganti. Updated October 15, 2026.

**Research question:** which terrestrial crop profiles resemble an explicitly defined greenhouse scenario for a Mars mission?

This notebook does not predict survival, yield, food safety, or Martian adaptation. Crop Recommendation contains terrestrial labels, without Mars germination or biomass outcomes. Similarity is not a probability of successful cultivation.

## Run instructions
Run all cells. Both the Kaggle CSV and the original NASA PDS Mars files are embedded with SHA-256 verification. No added Input, internet connection, or GPU is required. Set CSV_PATH to use a different terrestrial CSV. Dependencies: Python, NumPy, pandas, scikit-learn.

Workflow: load and audit → remove duplicates → reserve terrestrial test data → fit scaling on training data only → compare hypothetical greenhouse scenarios → flag extrapolation → audit real APXS soil chemistry → sensitivity analysis → export results and provenance.

**Two distinct evidence streams:** greenhouse rankings use hypothetical scenarios; the real Curiosity/APXS module reports measured chemistry and a feature-compatibility audit. APXS observations are not substituted into incompatible Crop features.

In [1]:
from pathlib import Path
import json, hashlib, platform, zipfile
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
SEED = 42
CSV_PATH = None  # Example: '/kaggle/input/.../Crop_recommendation.csv'
OUT = Path('crop_mars_outputs'); OUT.mkdir(exist_ok=True)
DATASET_URL = 'https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset'
FEATURES = ['temperature', 'humidity', 'ph']
# Rainfall is not irrigation. N/P/K are excluded by default: units and
# analytical methods must be harmonized before adding them.

In [2]:
# Public Kaggle dataset snapshot embedded for offline execution.
# Source: https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset
# Snapshot retrieved October 15, 2026. Retain attribution and check the source license.
import base64, gzip
EMBEDDED_SHA256 = '54a5a6e5408668e668667efc50de2fc867c1b875e0431b4f54dd331b0a109a4e'
EMBEDDED_CSV_GZIP_BASE64 = '...'
if CSV_PATH is None:
    payload = gzip.decompress(base64.b64decode(EMBEDDED_CSV_GZIP_BASE64))
    if hashlib.sha256(payload).hexdigest() != EMBEDDED_SHA256:
        raise ValueError('Embedded dataset integrity check failed.')
    embedded_path = OUT / 'Crop_recommendation_embedded.csv'
    embedded_path.write_bytes(payload)
    CSV_PATH = str(embedded_path)
    print('Embedded dataset loaded; no added Input or internet access required.')


Embedded dataset loaded; no added Input or internet access required.


In [3]:
candidates = []
for root in [Path('/kaggle/input'), Path('/content'), Path('.')]:
    if root.exists(): candidates.extend(root.rglob('*[Cc]rop*recommendation*.csv'))
if CSV_PATH is None:
    candidates = sorted(set(p.resolve() for p in candidates))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Set CSV_PATH explicitly. Candidates: {candidates}')
    path = candidates[0]
else:
    path = Path(CSV_PATH)
raw = pd.read_csv(path)
raw.columns = raw.columns.str.strip().str.lower()
required = ['n','p','k','temperature','humidity','ph','rainfall','label']
if not set(required).issubset(raw.columns):
    raise ValueError(f'Missing columns: {set(required)-set(raw.columns)}')
data = raw[required].copy()
for c in required[:-1]: data[c] = pd.to_numeric(data[c], errors='raise')
if data.isna().any().any() or not np.isfinite(data[required[:-1]]).all().all():
    raise ValueError('Missing or nonfinite values: correct the source before continuing.')
data['label'] = data.label.astype(str).str.strip()
if data.label.eq('').any(): raise ValueError('Empty label.')
valid = (data.humidity.between(0,100) & data.ph.between(0,14)
         & data[['n','p','k','rainfall']].ge(0).all(axis=1))
if not valid.all(): raise ValueError('Physically inconsistent values: audit rather than silently removing them.')
duplicates = int(data.duplicated().sum())
data = data.drop_duplicates().reset_index(drop=True)
# Identical profiles with conflicting labels are not silently removed.
conflicts = data.groupby(required[:-1]).label.nunique().gt(1).sum()
if conflicts: raise ValueError(f'{conflicts} profiles with conflicting labels.')
print({'file':str(path), 'original_rows':len(raw), 'duplicates_removed':duplicates,
       'valid_rows':len(data), 'crops':data.label.nunique()})
data.groupby('label')[required[:-1]].agg(['min','median','max']).to_csv(OUT/'crop_audit.csv')

{'file': 'crop_mars_outputs/Crop_recommendation_embedded.csv', 'original_rows': 2200, 'duplicates_removed': 0, 'valid_rows': 2200, 'crops': 22}


## Explicit, editable, hypothetical scenarios
The following three profiles are example inputs, **not Mars measurements or validated optimal growing conditions**. Replace them with documented greenhouse/chamber or simulant-experiment measurements, recording source and units. `humidity` means air relative humidity (%). Soil pH requires comparable analytical methods and matrices.

Pressure, CO₂, radiation, PPFD, photoperiod, salinity, perchlorates, gravity, available water, and substrate composition are not represented in the similarity score. Results apply only to the compared features. Mars outdoor measurements must not be treated as greenhouse conditions.

In [4]:
scenarios = pd.DataFrame([
    dict(scenario='example_greenhouse_A', temperature=22., humidity=65., ph=6.5),
    dict(scenario='example_greenhouse_B', temperature=18., humidity=55., ph=7.0),
    dict(scenario='example_greenhouse_C', temperature=26., humidity=75., ph=6.0),
])
scenarios['source'] = 'HYPOTHETICAL: replace with documented measurements'
# To use your own data: scenarios = pd.read_csv('my_scenarios.csv')
assert scenarios.scenario.is_unique
if not np.isfinite(scenarios[FEATURES].to_numpy()).all(): raise ValueError('Nonfinite scenario')
if not (scenarios.humidity.between(0,100)&scenarios.ph.between(0,14)).all():
    raise ValueError('Humidity/pH outside physical limits.')
scenarios.to_csv(OUT/'scenarios_used.csv',index=False)

In [5]:
train, test = train_test_split(data, test_size=.25, stratify=data.label, random_state=SEED)
scaler = StandardScaler().fit(train[FEATURES])
X = scaler.transform(train[FEATURES]); Xt = scaler.transform(test[FEATURES])
y = train.label.to_numpy()
model = KNeighborsClassifier(n_neighbors=7, weights='distance').fit(X,y)
pred = model.predict(Xt)
metrics = dict(accuracy=float(accuracy_score(test.label,pred)),
               macro_f1=float(f1_score(test.label,pred,average='macro')),
               train_n=len(train), test_n=len(test), features=FEATURES)
print('Random terrestrial validation; not validation on Mars:', metrics)
labels = sorted(data.label.unique())
pd.DataFrame(confusion_matrix(test.label,pred,labels=labels),index=labels,columns=labels).to_csv(OUT/'terrestrial_confusion_matrix.csv')
# Distance reference: nearest neighbor of each training observation, excluding
# itself. Heuristic threshold with no probabilistic interpretation.
nn = NearestNeighbors(n_neighbors=2).fit(X)
threshold = float(np.quantile(nn.kneighbors(X)[0][:,1], .95))
lo, hi = train[FEATURES].min(), train[FEATURES].max()

Random terrestrial validation; not validation on Mars: {'accuracy': 0.6818181818181818, 'macro_f1': 0.6698651715602864, 'train_n': 1650, 'test_n': 550, 'features': ['temperature', 'humidity', 'ph']}


In [6]:
def rank_scenarios(frame, features=FEATURES):
    # Each ablation fits its own scaler on training data only.
    scale = StandardScaler().fit(train[features])
    ref = scale.transform(train[features])
    query = scale.transform(frame[features])
    rows=[]
    for pos, (_, s) in enumerate(frame.iterrows()):
        distances = np.sqrt(np.mean((ref-query[pos])**2,axis=1))
        for label in sorted(set(y)):
            d = np.sort(distances[y==label])
            # Mean distance of the five nearest profiles for each crop.
            distance = float(d[:min(5,len(d))].mean())
            rows.append(dict(scenario=s.scenario,crop=label,distance=distance,
                             similarity_index=100/(1+distance)))
    out=pd.DataFrame(rows)
    out['rank']=out.groupby('scenario').distance.rank(method='first').astype(int)
    return out.sort_values(['scenario','rank'])
ranking = rank_scenarios(scenarios)
q = scaler.transform(scenarios[FEATURES])
nearest = nn.kneighbors(q,n_neighbors=1)[0][:,0]
audit=[]
for i,s in scenarios.iterrows():
    outside=[f for f in FEATURES if s[f]<lo[f] or s[f]>hi[f]]
    audit.append(dict(scenario=s.scenario, nearest_distance=float(nearest[i]),
        training_threshold_95=threshold, outside_features=';'.join(outside),
        extrapolation=bool(outside or nearest[i]>threshold),
        biological_validation='ABSENT', unmodeled_mars_factors='NOT ASSESSED'))
audit=pd.DataFrame(audit)
ranking=ranking.merge(audit,on='scenario',validate='many_to_one')
ranking.to_csv(OUT/'similarity_ranking.csv',index=False)
print(ranking.loc[ranking['rank']<=5].to_string(index=False))

            scenario       crop  distance  similarity_index  rank  nearest_distance  training_threshold_95 outside_features  extrapolation biological_validation unmodeled_mars_factors
example_greenhouse_A     lentil  0.153033         86.727791     1          0.170793               0.305011                           False                ABSENT           NOT ASSESSED
example_greenhouse_A      maize  0.157037         86.427641     2          0.170793               0.305011                           False                ABSENT           NOT ASSESSED
example_greenhouse_A     coffee  0.237255         80.824053     3          0.170793               0.305011                           False                ABSENT           NOT ASSESSED
example_greenhouse_A       jute  0.292827         77.349841     4          0.170793               0.305011                           False                ABSENT           NOT ASSESSED
example_greenhouse_A     cotton  0.364482         73.287895     5          0.170

## Sensitivity and validation limits
Does the ranking change when a feature is removed? The analysis reports the leading crop in each ablation. These checks do not tune the configuration against the held-out test set. Terrestrial evaluation uses one random split; without location, date, or experiment groups it cannot establish spatial, temporal, or real-world generalization.

The index `100/(1+distance)` is a monotonic transformation without agronomic calibration. Extrapolation is flagged by a feature outside the training range or nearest-neighbor distance above the 95th percentile of within-training distances. This heuristic may miss domain shifts.

## Real Mars data — Curiosity/APXS
Official NASA PDS snapshot: Berger (2020), DOI **10.17189/1519050**, sols 0–2301, Gale crater. Original CSV files and XML labels are embedded for offline execution. DS01 contains target analyses; DS02 contains statistical summaries. Rows are analyses, not necessarily independent samples.

**Material selection:** the soil subset requires exactly `Strat_group == Soil`. Sand and regolith are exported separately. Rocks are not automatically classified as soil. Records marked `Low quality spectrum` remain in the audit and are excluded from the working soil subset. Other comments and analytical errors are retained; this does not replace a full instrumental quality assessment.

**Crop compatibility:** APXS measures bulk composition, not plant-available nutrients. P2O5 and K2O are reported in wt%. Stoichiometric conversion produces total elemental P and K in mg/kg; it does NOT establish equivalence to Kaggle P and K. Do not infer N, pH, growing temperature, humidity, or rainfall from these APXS records. `SH_Temp` is sensor temperature, not crop temperature. Without analytical harmonization, real APXS chemistry cannot generate a crop recommendation.

Source: https://pds-geosciences.wustl.edu/msl/urn-nasa-pds-msl_apxs_supplement_sols_0_2301/

In [7]:
MARS_SNAPSHOT = {'ds02_2020je006536rr.csv': {'sha256': 'c400d2fcb07835ea5628d28b3aca891bce25275fc04d8df0a7fdf468700cbd0f', 'gzip_b64': 'H4sIAAAAAAACA7VaXW8byRF8P+D+A9+SHBaLnZmer8tT7oLgcgdbRmQkQF4C2qJtJhJlUNTl/O/TVT2zpKjVIUpsm6LF9bJ7Pqqrq3v28nZ7PVxuL/zwGm9/uPYXYfjT5mJ4sdOf9xfD9+uL4eXaXww/6c8rfxGHS73j++vh+z1ufbn9x8ePN8Pfd/znuz3++fqrS7W6+nb18v7mzWa/un23er3ev98c7ob663/nb77Y7rY39zeD+NHXYRprHsqY0+DyGPRjKPrRlSGN1Q1+TKLXBG8lDjImr7/xLdRBQhh8zoMPR+PrX8x4GEsY3OjU8yhqaBr1RloqMJxHn9R6STCHGycHj7ilRn2LfkiShlD8kOPR+ma9g2nn8I2opl0cXB19gWmvpmNV05MOacyYDExVP8SxYMg2gjREnV7wQS2dWL7amm0122x7mM7dcjpaxiWONMyWMQ8pajkPwU2nlv+y3r3fqMmINfRcvjgEMzDhjQMNsf+vw544LA6vBfyv03XWH6fDin62fHlY767W+6vVHzc/b9eH7e3OBtHsqjWHjdSPsOtxTRdOP863TPCAT87bp5QHmQanKz6c/VGv6u0LwFmtLsJZ8fWc12xphrcDGHS1i+4bIJZsHYLoZwVBGiN2OvMalkU3uEUEd0ptKsr0szux3dFd8BVdXdE9GYuCQlFRDRbOY6kznAfbUf3NKVAigcI3vRCqDD5q1E3haJ7wFkQOwVXpoCLyJgRRNXgDDWZNAvcvWADYJYW3RpCfwhDSiWWDt4wxzbYRlMQCTQvWxcNysEuMF421FkY6tejwHZ2Xmw0bupPNSwBsL/jYQCUIDw0nQQBgVYuttt4WOrixPJMO1+myFZktL6DbjSl1EOfYYwpfNXM+9TdDd7Vdbb+RKnTwgTu6gO8Pm83HN5ur1c0GWPwCUO8OFuEenvV6YO0E8lMiItKokazI4Zo7Za+AvVBq9yDaiUTPlSm1A0eRLkp0WSHL/T0x31GvoM5AvaJFo8gHhJXayggCHVUeFdPBCIz3eSAKu1ONi6uOoupwYi4PHTRaD8b+Bbj2jFwOq2I70xhmUierC8CV+tYrXhO+qj69pDPrndoJHDpAgGDgtF+Qd5IlpBRnB9Fi1gHRzAFZCTIrgvyUHzjoMRBtZT3WXhCljccn4B2hbmwsFrgJa+gs0HRNpE6EZ5SHxp8OA241CAe/S6f+MoeZgR4Ul432yPBgNBBf8ksx8Or2sL67275dbXZXXyoM/rbdXf1zvVv/byLm5Nsd9qF25ERIE8VlaHgDNwgTnAcgCddMjiudi4OKDF2WIGD6kh466MC3dR6BZ2URnxhXGj/K9UUAEwSWwzYToKo4EgDjDE76XVHKl1R0N2o+mwSwrzOo5MlsX/RikgvpAXwcRw+6k877BdNCFhldNvCLfjUUDQrnJ3/mwOA/GQtPQJ5+X1cpZmPfClxGsLfDIjpLZQqkBBC7scWXqK5RPVYGJ9U9cNElDgfDkeqYg310SIOFjK3fxHJ5phSPy2K5iJTuMgaPgMs1PDC/qHPCzPAZGKdWmtKMdteEDcXkBM+59Ou6BDo+UTWQoluKghf37zfr/fZwGgfffP5A+HH9r82L1dtrjbjFWEhP/o14PzcwZ4ECRaJbpvggjqqJTYelV6aLUKvREijpXyh8ZmaFclAu0v1Hmjnz0SIiNpBk01XqySGSmH21SDAFkLGtHkztwEhNWpsErUPSNKLMWuWRD0REbOJUA8hFIEkDIZ3gJoG4E3DmTWRlTIKIrkZ0yoZB55Afr1MLiGrSHWEcwdGuFQiecQz0B9QkYHDK40zZZeHNYFK+Dwrvks89WDx4u9GL8UBbbJYuVvEg4kOyamNqQUe92Nhd3eeAhTq3v5gTaMQCgtPKPS0YB0XThk3c1h4LTNYYCSo53S1V/t+cOrvavNvuNBG9+bS6fPvhZnt1WG0Oq/X1uPKTk9+vXn+4vfl4d7s7uZoeB9RSiO3360+rd7f7G5vAZ4+uV+sP+/ubj6sfttfXy/GlXPOc12OTcwKKJpoTRJYix5t0saQM/DAXSD6ppN1pog4DSgjFuCz4aBGXBaLBQVixrPAE6gTRUaCD9WoC1aZExvb4D8U1yxTmftC71xDyoviocVpwxcATC1gW7gjvlid8NAEpKPx9E49GvNGmZl0BXSkMwSP0XVpywuiLquXoJlA8CqgqiyVqQZITThKrybB01KeMbYaHxk1Epep1UUp47MZCMKi0Q2igpYEMnltxg44HkoKDO29BQC2JoQfsWVPS2DgkJi8C+bown4VYzBbL3tnOi8UiAz9iyaxOc1aSsPRvHMbSRbkm6GJDzXq0EX5Y7w83693uN3erv66vrzeflsuH6VmvZbNzBglkiQIeRwUaMQ2mUUGKCCB3y+KUvGBaK8Wsi6HLpTNxULz5CUc9jQQKVJ2yY+NJKxbuhQDUCcgC1ir2kN2MAk2HS8w1AYhG9VHUuSR5wtfGyL41mSpVQgaKJyS9hIuCjfetNsbMkWAYbhMrnKHgm0gArjzlpScVUv3EArvSkTdHEdEklrGYZ5kOSrA807KKFlGFyrUoVDGxMS77M3xbe4o516P0Mu0DN65nrXKUSCyiUi8BgccJb6nqvsoTO7UAb92QufygprPMky3QJqML037Jco2zuDWq88ivCRlHXf603q/v3tzv3y+CGmn/Ga+H5mYwi2lFHbQFfaGgJZiR6QvCj50h9iIBht7NIxslzckhT4P4M/vHXpBtOQgzAvtGMgEYUlhlWGoZoaaW8tl19QgupgQ3FTRBFHYFnawHbgjfZOocmYKkXK23ZbqdzSVsOOdJDNTCZo6zLWDZrZznoSnKuX0DrkU6clYZ2TogbukA2c96B+agmmrPyF6uNdXUg06zQErK9NCDQVUs17GgEAK2dWwYdq2VkBq1T6a/vJtbPpkVTtYx5nq20YtqKByr4hmhxWRR++jdXCOzPJCmjVAlsesHNYQ+y/3hoPrmz3fXT3UqdazPeT02OeuI0DrCmHw2XdqwGgAAj99a+FrKZZi39bI2QlR6hzBNC26OrCukVJCpIEkShxldywqEJWetS7ZDPbchsMtpDXoFnBOAWtDcE2STR66I2mgtVqI2st1I2eCBp4p07jAdMRokezRMxYZaVpvIv1IXnczQzR26Gky+xS7PMCbUCyzPg3XGIOlBzJaA6UM33IE+/cKaNSk/WQZkgxMdgNZLSpiEFCtEnOGVDc8EbQE93RL7UDUivYMaC6h4zt0sYFgguXoT0/UIQZBSBU4tgNoZQbDfYlP9DkcTAW0BZa3vru93h7vVq9vt7rAsHtyzXo8szvjNVsAUtpI98MtGi6MkphDGRtUyN6vm8i22KA8K5IqFW/DS4dvONSrLy2B8xC5/wUWoNctLkBWUwcIeOUMLXItDHIF4DDU99tKR66wfU9B1aTzqMxWJgPA9s0hpFW22ao6EA0xhjAl0m6YFDx22IVveKCzBHd4n65+7ZNm8eUl2X2ztpe4FLUM2+3N+5KX1ZMR6jmw1FniyooEtJbZnc2uU8JxPV9F5CxcqO+ijgBhHkRxAXWdeFnAbWi+mtO7kSdmLQVA7uNLFrwN02a5jhQroCvKtLnrVTPhqczhs3u9vf978Cnx11M95LVo9LeXEklxmXyPZIcuUO115at/QC56UG4VJy2dJZ5jUkc/Ljo4kXIslaMfD1ND2QLw16DLcBetdsZJICu5k2G6FyYQ40RtijMu+unYInR4Lute2/J6NS6A5MgnXrkErz8YAOU4JQCsIGxwRxydWb4Y0E4u5KuyxG0dxVcVUSpx7KNWa5s6Wk44gE93kIzTw8qQasIuV0gFHVAE4J7BZS6XYKkU2/6RYR9MOt4hNXXNBLwokEP3ylBaFrx0mtb57y7xT74O1k9vST6iYyfIsa3BMhS4LUh0k0o/3++UWuHLKc16zpRMIc+Nszqh4uLxoKKER65FIvAkLBnlEirQOV7HdCpobAwTqcZgzbAUY55mII2B9I3ABCSqBV+Muu6ndmExk2ym9bh0opcBPPXFArBarlWuxA0l809mGsT8tYKhTrOLxAMMcpWmwliIHF05MGzqL9ZppHLBL1oNxCARhJufhjevI4TMQsTXA1bTCumIXJzebnsFYQpcHwp4JURHwOaRZHsynbhnlj4EmtpNJwWmzoE2aj4uySK+kbBOzwOD8iQ1dX2YMtmaCtyUlx2L0oAD1n9x/05b7dqWFYDsQvfv8Pbpzd8vqRPPR//Gz6OVccT/ZuSP9Hjt3JJMnOndLbp5o3p1Ibqsej827prlnfHbdEi1uTjT3kr8WRKxoAFfb8fnpkmidtRj7MX/J/dEbKnBTk67pbugkVECLjiykWDA0wq+srU03DJT+wmNiS6h80qeyqx7syQnzlIzvrR2w4KlVj71T3jp54LhkvFso512rhartD/mNstMey8CpEgooQUEi6EsuuFqItjj35ByitB1ZUcw4ZkiZ23bteKk9MRXb4zUeIhyNo7IYb5eH7Q2658dh/Pbf28OH2/vD6sP6+nZz97sv8BhC8/l5nkKYjc0hVVpzGD08qOL5ADZBEwhwYuco/nhYEVt/CJLZG0lBqJ1YPz6DENrDXwglDZVWYwBG9sQB85lx7ZxKWHYw4oakSb9oNGbvHzhozyDMj8ewG8nY4xNxEWKFlMCMJC16YoNz61qK7ntAdeDcQ+P9EYTePKm93Sn2KF0B8SQ7isvHgwK0X0K10rI5QA8HZJBPHViUFJOkVHaeutIyerDHclok2vNmzvakWL1qx0NaA2s6SXqX/pyaX34IIbQnCizKjw+UcY52aDA/ilO65OjFQZ5QVuChn6+/+g+UrnskKSkAAA=='}, 'ds02_2020je006536rr.xml': {'sha256': '47bf8224a5deaf5fd36a00ef213b31c2fef8f9e5f21fa1361809cf0df33c7a42', 'gzip_b64': 'H4sIAAAAAAACA+2db2/bOBKHX7dAvwPhV3dAbP1J7MSC6kWatHvZTZogzhVF3wi0RNtcSKJA0mny7W8oO4nsyLnKidndZgzUtS2KHP44Mxw9cuvwt5ssJddMKi7y9y2v47YIy2OR8HzyvvXfq0/tg9Zvg3dvQ9OunYmEpWQq2fh9a6p1oQLHKRLVyaminYm4Nm/2zJNz7TkXx8O9CJ4i78h1Oyqetsi7twT+ZhnVtwXL1bwT08dMph0Rp3FHyImTqCR1Fu2kyOfjX0iRzGIdnY8Uk9dUg7k0JWBUtZs1prTIcx5gNDzKkQLob8ujbfFRnciN4vcT+f79e+f7bim977qe8/XsdFiq3+a50jSPWevFBlc8mK/sqYjLRfy/chLS0NNuVNIa1I4eniQs13zM50NHh5LRgTXhSwtSMYHR04gvLGFyMJN5YKZlnCvIVBrR4kZFalYUKcugVaREqiI38nddL0iopkGiXD/yXd/9i7lur7vbkzJ0anp+0pJFzEP7AUR96FTeW9BBc52ywRAmKaSGXENO8rGQWbku5PwGJkEumZqlWpHhLMuovCVXdJQyAq3Ix7kwEP+xyAqheHkWzRNiXMuoQNi1SGflx2JMJkyU6hDon0lOU0V4Tn6n0F0szUc75IxKFZDDi69DIhfjjqXIyAdJk9EMRj+F7o2dWhA9ZeQLk5RczkbQzyVPJix05jN6eaX4gzBRmX+jxUrBsnl+xzVrt77Ns4cvFnk3TqlSg9osHDrLjV5m8qFTF62LY5XxrUXxkixXPGPRkRASnAI8SA0ejpXHIW9KHUG0skhD04Hven7b7bd9/8rbC/zdwOt9C53VVo86EcVyH/2267X93hV04HuB55d9LDWq2Og8YWR4kl8zpfnEooYrs8spmGvijgxjDjUHgxgbCYhHIW9DpzxqzRZTjwzOuDIxA6Fs3llU4iSHHASBFF2yMZNGCoujz7clnkTyfvClHSkWYN2NDnjVYYJsrlUHMjPsVfP1i9LK+i332MSW+7Oich3MjhdpES2NHzorrV5kHRy7C1EN1dpwrDSYJzzYgKLhrdIsG2zfK1eHhEwCm20OOXlgySt/WoaosaX0smFBYwYlw1hvM0nUjG7RMR+P/uMZQmk5K0vWqVA6UPdqdaCs/bGUsGb0lWjnKloZ7GUSQo3yFnPCajZ6IgD/nqF6mBZTSi6gpuEx1NZf25LekmHBYriMzxjI+GJBuy5AT+69AgP0qQANzDXmj0flMyP0hXbrf1SE1u7ya3fx6pUFlROmo+WLn+3W5usK861X4WtL8YuU5kxbqMT/kaW4Ll0kKEqRyhK8eRg3K77nI75gHP+tim/rIXdHOB5hjMWBewWiU660HbixrMnHG6txUOMPieADz+14rt93DOv84+OcdYaOObLVoSteDuFmeeLm8YGBQ8od8keHHHYI04SmHfIvo8K/d7ZGPx+bsTkOBdPFzPiPMeB3JorprSotu2SKURlPAzJP8WqHfAFbM0Y8v7tDTpSamZc7NaLAsgd1DtF5Vtb5aWsdOj8hxmzmkdD5SWmsYsInnrIysy4T6yoINU1Wr6fG5rSy9qm51dKJ1XXoPDRZOTeG0cqEXgW3vtv23LbrX7n9oOsHrhs6Ne3WOcqKieF/GE2YHNSVbUcmmHJiWkB43lVwKy3FeKwgqcxyrt+3RreatQZg0PzTR21Hf8H1W5SyfKKnS6ccdOGc6tHVUwvIKqbWNfcQEyoTc2vp4nhIjodfiBc6dYdX9wCmYsmLcjuG0jqjbcXgNBAsIWZiyqSXuJxymdRMKjIlC9HmXlEHtopKB1VFHykYlneXomOW8oxD97XiDgVPybxYuLsr9cMCH/TWKfx8mSSLhUzUYN+ks/nL2hZRspieHBxRCVvBhLUvmYZETU55ztqfGEvuuqi0fRQcLF3qyiyMCYjlj1fOupz3uk7gh57VwNtf9KbqGk2kmBXKOOzi1bu3b+ABcWzGf6r/h4X8DE+1K7cyyXyWjWAq3t3sFu/XnDMvlk1dfDg8OjmJhlrCwoET3n++7sSKlx4/vDbOXbrcsXHpoYZ0oTSP13l1NV8sS1FX6DeRa8jP/QZy+ZvJdcnMrcRGYhnDyBeazthGojT2m6tmQuxaE+LKshCHqX++20CJPWtKlJbZlOITO28gRNeaEGCXTRnO8iYy9KzJAHZZlWHSRIZ9ezJMrMpwRJvIcGBNBrDLpgyfqd9Eh741HYxhNoX4s5EOnmtNiD/t6nDhn3ebCOFZE8JYZlOJYaP6wbNYU9otH47SJjLYqyiP0meoQEhzHWSzitKzV1KWptlV4zOPiiJrIseGhaW56TIxN+abbR/crhzf8qZy9GzK8S23K8cH2VSOfZtyfJAby/E0q3+CIZmbeKsA782buz/bIKb7Xh+Z6Q8wU3NjalNm6nYRmiI0bQ5Njc8hNEVoitAUoSlCU4SmCE0RmiI0RWiK0BShKUJThKYITRGaIjR9AppuA5h6e91uc2La7706YjplrBixjamp1+36iE0RmzbHpneOh+gU0SmiU0SniE4RnSI6RXSK6BTRKaJTRKeIThGdIjpFdIro1DI69X1/vzk69cx/M/C62OmF0FQpHm/KTv3d9UIjO0V2upad3jseslNkp8hOkZ0iO0V2iuwU2SmyU2SnyE6RnSI7RXaK7BTZKbJTy+x013W9Tdip/9rY6dlswqjkenN6uuu5e0hPkZ42pqcV10N+ivwU+SnyU+SnyE+RnyI/RX6K/BT5KfJT5KfIT5GfIj9Ffmqbn+73Nvnuaf/g9fFTaX7veMFOL9mk/Kmx5hC137PwU1F7faSovx5FLR3w3vMQpSJKRZSKKBVRKqJURKmIUhGlIkpFlIooFVEqotRnotRN9o5flaNuoMUvC1E30OLXJ6gHBwe7GxBUz33dBLXxr0X1+/jtU+SmG3NTxKWISxGXIi5FXIq4FHEp4lLEpYhLEZciLkVcirgUv3mK3zzFb55a5qb9/d5G3LT76n4xSvNMiXxTcNo/2O8jOEVw2vwHoxZ+h+QUySmSUySnSE6RnCI5RXKK5BTJKZJTJKdITpGcIjlFcorktJac3o0BHx9KRqPzkWLymuryX+/C8dC5kCKZxfrRkf8Bym4X97bHAAA='}, 'ds01_2020je006536rr.xml': {'sha256': 'c7b752b4bdb4041594ec7a1d6042b8efa11d47b5af989ede2c7ebf95bcea6ccd', 'gzip_b64': 'H4sIAAAAAAACA+1cbW/iuBb+fFfa/2AhXakrFRJC6UwRw6ql7Ux3Sls17OxovkRpYsC7SRzZTl/+/R470AYaujVD7Vndy4eZgh37+PFzjp9jG/q/3qcJusWME5p9aLRbbgPhLKIxyaYfGr+PT5vvG78Ofv6pL+s1UxrjBM0YnnxozITIec9x8pi3spCHrSm9lW/25D/Obdu5Ovb3AvgnaA9dt8WjWQP9/BOC/3EaioccZ7xsRLZRsKRFoyRqUTZ1Yh4nzrweo1nZ/xWjcRGJ4PKGY3YbCjA3TBAYVW1mjSkN9D0vMBpeqqcetPfGvb3hqzqQe04eB3J3d9e66yjoPddtO19H575Cv0kyLsIswo2tdc5Jr5zZcxqpSfxHOBHSZNo9jxuD2t77ZzHOBJmQsuvgkOFwYAx4ZUFCp9B7EpC5JZgNCpb15LAkuXopT4Iwv+cBL/I8wSnUCjhNeOAGXsdt9+JQhL2Yu+3Acz33T+y6+93OPmN9p6blFy2Z+zzUH4DX953KewM4CCISPPBhkJQJiDXoLJtQlqp5QZf3MAh0jXmRCI7G4U2CEZSikxIQ8PuIpjnlRNUOsxhJSsnRI3xLk0J9TCdoiqlCBUG7mJEw4Yhk6GMIzUVMfrSLRiHjPXR49dVHbN7fhNEUHbEwvinYAzqH5qV9giIxw+gLZiG6Lm6gnWsST3HfKUeyfYTIEyCBirvBfIZgutpey5Vztr7Od3efz+NtlIScD2qjb99ZrrSdwfedOi+dl1X6N+a9S7CMSYqDIaUMSAEM4oOnMlUO8ZKJALwUBwKqDjy37TXdg6bnjdt7Pa/Ta+9/6zurtZ41QvPlNg6abrvp7Y+hAa/da3uqjaVKFRudF4zsn2W3mAsyNYjhyuiyEMyVfof8iIDWwOBjNxT8kbKHvqNKjdkidchgRLj0GXBl+c4gEmcZxCBwpOAaTzCTUBjsvVyOSBywx86XVqKIgnX3okeqhOmlJVatFOYv4OX8BUll/pZb1LHl8alAzYNc6QJBg6X++85Kra3Mg2N2IqquWuuOlQplwIMFKPAfuMDp4O1ZudolRBJYbDOIyQNDrLQWIWpsUSzz8zDCIBkm4i2DRE3vBon5vPfXRwguWKGk6oxy0eOPaLVAzr4uJKzpfcXbCQ9WOttOQKhB3mBMWI1GLzjgj+mqh0k+C9EVaBoSgbb+2mThA/JzHEH6nmKAcWtOu85Bzx5Z8X8HfclBezK3fL1XfqeHbmm1/ld5aO0qv3YVr2YWIZtiESwnP2+rzdcJ8zdX4Wul+FUSZlgYUOL/SikuFEV6uQJJSXB9N9YT32WPW/TjH0p8G3e5xQ7Hs22MecEjAsE54cLM5sYyJif3Rv2ghg8xJYO222q73oEj9zh/Oyn3OPuOLHnTrissB3czPHD5OsJASLaLfmuhwxbCAoVJC+1IFH7ZfbPdz+dmbL4dCqbTQvJHGvAR03z2wJVl15jjkEWzHipDPN9FX8DWFKO2191FZ5wX8s/dGlBg2nt1hGh9V9SxNtd9x4KPmY0kfcdSIHuy4JQkWIXW5S3r6k6orLKaUE3kY0r81JyxtCJ+23eeqqw8G0FvKqJXd249t9l2m643dg96Xa/nun2npt4aoqxY2P+EwxizQZ1sG0pnypCsAe65UHArNelkwiGoFBkRHxo3DwI3BmBP+emzujd/Qv4WJDibitnSI13Xg4eqxavP5hBWpNiVh4dxyGJ5pnR17KNj/wtq95264tVFAPOIkVytx6Ct07DJMTwGgMVIjozL+BKpMauoJmOR1CxIyMOiFqwVlQaqkD6DsK+Ol4JjnJCUQPO16KqAWHcq5VOIqq6MhvJ07tWod93OOty/HzuGI8piPnjX9mSYK9/U1gni+ajZYBgyWCKmuHmNBQRwdE4y3DzFOF40Uan7zGdwstSUnC/pJ8sfrzx1Xba6Dvenlvmg2523xusqTRktci6JPP+rps6psuWlvp7mGqa0dh5XxpsV6Q2Mqr0Y6Pz9mmdKPS2l86E/PDsLpOadyn2Rp4J1T1aIDKahEQ55wZQSQL48RpJTVM/2aiD5p/FrQTSeZwWvRsnbDCVfMOC6JkilcegCbDMOjB+mOQQTkGZYA52OQXQ+4gwkW4LmKC1uYRhH6ghzEcSMJDrOtmcQKBXyq84GC85oLqSRHOMtCJxYRv7hDI+IEuIOZcg/HJmnnYCVMVDhTwPNrkE0lYVkysJ8BgnJR2kpGlJIZkgms4lFQLGC2yRtaaC2bw2108f7MT8KcukN04HunTXoRlh2+APgNr/llAR34UNOSSYcWoiI0VxlFBpYvjcZChOVrwoIeWpZVXex/lgM4LIcgHEoh2HTDzIqMNeA7cAgbBfSNnSF2YJzsFack79w8oCGYRKRIm36RTKBNaUkZiov69hYi6VghxWu3IjNGRZzM14vgl2DsF5CrscW+mVuOjeO2RUr81F18w6LGY118NLOGn4fn77fMBAqSYoq9ir3rSgb49idEhGMwXwdxExmEOXJNci8wyxMHjjh6BqiNCTE6DDPE7JJwiX3df4DL81FVt4SlJf5dJAymU1U9fH5yB+XGekmhNIG5xy4LIpYC5oN84drLO+6bpKELoxEO8d4yjDmv2xAnU3AodlUG52ucXQWVpqG5yTBt9pr3L5peB6tRDsjeYnGFDoXlImZigWvB+edaXAWRhrG5iTkQhOa98Z5U9poGplNfOrgf8WnfLldTycTnT1T1xg4J0CYVJ2t+E8q8QiLO4yz8jBYJSiRgPKMg3qUh85zRP2CTcIIQniUmsLyUzDGqc5Ol9c2BuXhLWbhFC9v0yvQmvLoCUnLoYaAMrQzxAknhTFRQCZY6IlJz6TsPi7mzJM7rRX4dj596o1GPd83hNPpH59GOhh1jJGrTEyKVB5CLi567OAvKBSo2zpAf+EvpiRC6F3qQGROeEvLZLSKgDlzOu3cif8axCXAjOlgY052XzEcEfntHXTCGERxkqHtwbUBWqOpFonMCXAwzB6HoHNdCr2zSSGrYB0m3mVHBypzalyZZhkZXSId2CSSZcB8cunpHNqbk+fSMru4aPKo07bJI7twXXmXXR2oPHNQgWV2cdGlUccmjezC5Wstax1zAtu3GqO1l7SOVXltFayhztWqjjl1PUxsYqLLH6va2iZUn7VS/I45Yf3ZZob/WTvB71gV1VbBGoY6FNozp6jBMKuoaFJoz6qetgrWWC8r2zMnp8dW04yxfla2Z1VO24VryPT2ifbMCWplmmVkdJlkVVRbBmyUaS1qBnets0urqOiSyO6u9RbBQkj/jBFrkcicuAbD7JEIOtclkVVxvTWwNuLQBdH5so9r8ouKF2QVlzw3dT3kgmhyqGv0O5x1h7BbQmsjFn3TubDV9UxC9S2zx6JvmS6LOpZZtEW0NqDRkRZWeyaxOmL2aHTEdGnUtUyjraGlOLT6qysvfOtf/hxT3S8xlD+DUf9bHn2n9oepoeRvnmiATNhgAAA='}, 'ds01_2020je006536rr.csv': {'sha256': '6a6e4359a543b315783bc672b9a173e4930592ba62c071d5c93b9a7b85584f8c', 'gzip_b64': '...'}, 'readme.txt': {'sha256': '735c60c56a850e62b198956e7e3aee0fd9189469d1b5bb57d12a50c0c6c9a49b', 'gzip_b64': 'H4sIAAAAAAACA4VU224iRxB9X2n/obRPIMEwYONkeTPYsdYBL/J4N86T1TNdMB33dI/6AstbfiO/ly9JVY+9sI6USAhNV58+VXXqMke3RTeA2wwuM+hN8knez+DzDt1O4R7sBlbFEi7XjwU8CIIGDxtnGyishhyChclZPs5gfVXADVpfKTQVerizEjOQVs1gnGfjn8Y/fxyNp+OP+TR//+79u4daeSijkRqhsiYIZTz42LbWBWW2oMzGukYEZQ3QF4QaoREm+sqpNsyY4sO1xgbpqSaGprVeJbQwEqoaG1XRBe6sjslMeWzRartVFREFdEpoT27gRnAIjk0DWAnnZ12yDn3Ur8nOnZBldAdYEj3HR3lzSF/RCbiPJfHcK7nFD3CLmw0LOU+yQm/cH8B9RtJojS5057sMPhHCHmR3XmewzOBX5u1N6LxKpVhVCyfUtoMsMriiqvz9518LawyRDRfWtuzgrHtwnUFR1Y2S5OO8n8p5Q6bWiUOHIQ8Ee6hJKk96JFuRMe4r/YSZ26ZBDb0p2ck7Xf2OBLugZuh9roItyVlqDri10RkSlySlirf1wSet79GjcFU9g7UWhvok4ypR9PDFKGonqs8hvYmo25pSEkZIQYhJHy6jD05oJQzcpaIT3/HV4HhP8LO3hHfUpnMXjd+r6vmE97xPZls9/8D0/Xbah9+Er0n0QHKcMFIti0AVsVH5AXwpLgl8QVmvl8OF0AGr+sXa9TFSf2pt96kraod0VpoGQDhkqnBsdMZLn4+fWMY/MM8vpmcXzmWV30HvSgQBBQa4KvLxmw586f+Ncj6kcQNvqXkpdTbzeDbKe1KtnxxM/tfBhDIkmX1IhfOxaQT1NpP9u/WTq3zIbon+R+an48Q+nUxs1soN9IrjNH86mWZpq8hj22c5rkVVf7ek0ZUcZuusjFV4ox/Ql0TeACVKKA8geOucgxYlNS5hBXhsBY9yqgHsVahTVl40CIb/yhiOZvwW0LBu8LhaZidkXfXICD18JIwkf7QanmPLC2AbxRb7L2UOtQi8wTjB162QiE5zXtEy1InP82oSQG/qEFo/G41a6TMjvMi2dseH81GHGTnUKDzKEU8tHlJIJVIH8F6mgDjhFPlQ7PkuUDqAUgVLi4ylbGj0S4ToO7W83YQOmEKmt7RQVRs168WxvdahG9wCGUkzTQ+j45XOfdjY1NbHUjN0CL/Q1enKFqUloVmHAdUE/yPZhKQUu3H6By1vNJuNBgAA'}}
MARS_DIR = OUT / 'mars_pds_original'; MARS_DIR.mkdir(exist_ok=True)
for name, entry in MARS_SNAPSHOT.items():
    b = gzip.decompress(base64.b64decode(entry['gzip_b64']))
    if hashlib.sha256(b).hexdigest() != entry['sha256']:
        raise ValueError('Integrity check failed: '+name)
    (MARS_DIR/name).write_bytes(b)
mars = pd.read_csv(MARS_DIR/'ds01_2020je006536rr.csv', encoding='latin1')
assert len(mars)==712 and {'Strat_group','P2O5','K2O','Target'}.issubset(mars.columns)
mars['low_quality_spectrum'] = mars['Comments/Interpretation'].fillna('').str.contains('Low quality spectrum',case=False,regex=False)
soil = mars.loc[mars.Strat_group.eq('Soil')].copy()
soil.to_csv(OUT/'mars_soil_audit.csv',index=False)
soil_work = soil.loc[~soil.low_quality_spectrum].copy()
# Oxide wt% to total elemental mg/kg. Does not measure bioavailability.
P_FACTOR = (2*30.973761998)/(2*30.973761998+5*15.999)
K_FACTOR = (2*39.0983)/(2*39.0983+15.999)
for oxide, element, factor in [('P2O5','P',P_FACTOR),('K2O','K',K_FACTOR)]:
    soil_work[element+'_total_mgkg'] = pd.to_numeric(soil_work[oxide],errors='raise')*10000*factor
    soil_work[element+'_total_error_mgkg'] = pd.to_numeric(soil_work[oxide+'_error'],errors='raise')*10000*factor
soil_work.to_csv(OUT/'mars_soil_bulk_composition.csv',index=False)
mars.loc[mars.Strat_group.isin(['Sand','Bagnold sand','Regolith'])].to_csv(OUT/'mars_sand_regolith_separate.csv',index=False)
compatibility = pd.DataFrame([
    ['N','Absent from selected APXS dataset','BLOCKED'],
    ['P','P2O5 wt%; total P only can be calculated','BLOCKED: availability and Kaggle units not harmonized'],
    ['K','K2O wt%; total K only can be calculated','BLOCKED: availability and Kaggle units not harmonized'],
    ['ph','Not measured in this APXS dataset','BLOCKED'],
    ['temperature','SH_Temp is sensor temperature','BLOCKED'],
    ['humidity','Not available in this dataset','BLOCKED'],
    ['rainfall','Not available in this dataset','BLOCKED'],
],columns=['crop_feature','mars_evidence','direct_comparison'])
compatibility.to_csv(OUT/'crop_mars_compatibility.csv',index=False)
print({'apxs_analyses':len(mars),'classified_soil':len(soil),
       'soil_without_low_quality_flag':len(soil_work)})
print(soil_work[['Target','P2O5','K2O','P_total_mgkg','K_total_mgkg']].to_string(index=False))
# Earlier rankings refer EXCLUSIVELY to hypothetical greenhouse scenarios.


{'apxs_analyses': 712, 'classified_soil': 13, 'soil_without_low_quality_flag': 11}
        Target  P2O5  K2O  P_total_mgkg  K_total_mgkg
     PortageRP  0.95 0.49   4146.054765   4067.741381
         Yukon  0.97 0.64   4233.340128   5312.968334
       King_rp  0.93 0.48   4058.769401   3984.726250
        Barker  1.01 0.63   4407.910855   5229.953204
        Argyle  0.95 0.47   4146.054765   3901.711120
Johnny_Cake_rp  0.97 0.62   4233.340128   5146.938074
      Lagrange  0.89 0.45   3884.198675   3735.680860
  Sourdough_RP  0.85 0.45   3709.627948   3735.680860
         Kelso  0.85 0.44   3709.627948   3652.665730
        Dumont  0.92 0.47   4015.126720   3901.711120
       La_Brea  0.92 0.83   4015.126720   6890.255808


In [8]:
ablations=[]
for removed in [None]+FEATURES:
    fs=[f for f in FEATURES if f!=removed]
    r=rank_scenarios(scenarios,fs)
    leaders=r[r['rank']==1].copy()
    leaders['omitted_feature']=removed or 'none'
    ablations.append(leaders)
pd.concat(ablations).to_csv(OUT/'ablation_sensitivity.csv',index=False)
# Perturbation ranges are assumed, not measured uncertainties.
rng=np.random.default_rng(SEED)
perturbed=[]
for _,s in scenarios.iterrows():
    simulations=pd.DataFrame({
      'temperature':rng.normal(s.temperature,1.,200),
      'humidity':np.clip(rng.normal(s.humidity,5.,200),0,100),
      'ph':np.clip(rng.normal(s.ph,.2,200),0,14),
      'scenario':[f'{s.scenario}__{j}' for j in range(200)]})
    r=rank_scenarios(simulations)
    counts=r.loc[r['rank']==1,'crop'].value_counts(normalize=True)
    for crop,freq in counts.items():
        perturbed.append(dict(scenario=s.scenario,crop=crop,leader_frequency=float(freq)))
pd.DataFrame(perturbed).to_csv(OUT/'perturbation_sensitivity.csv',index=False)
manifest=dict(mars_source='https://doi.org/10.17189/1519050', mars_files={k:v['sha256'] for k,v in MARS_SNAPSHOT.items()}, dataset=DATASET_URL, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
    seed=SEED, versions=dict(python=platform.python_version(),pandas=pd.__version__,
    numpy=np.__version__,sklearn=sklearn.__version__), metrics=metrics,
    duplicates_removed=duplicates, units=dict(temperature='C',humidity='RH %',ph='pH'),
    excluded=['N','P','K','rainfall'], scenarios='hypothetical',
    score='100/(1+mean of the 5 smallest standardized RMS distances per crop)',
    limitations=['No Mars biological validation','No yield or food safety evaluation',
                 'No spatial or temporal validation','NPK units and rainfall period not harmonized'])
(OUT/'provenance.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
with zipfile.ZipFile('crop_mars_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file(): z.write(f,arcname=str(f.relative_to(OUT)))
print('Outputs:', sorted(f.name for f in OUT.iterdir()))

Outputs: ['Crop_recommendation_embedded.csv', 'ablation_sensitivity.csv', 'auditoria_por_cultura.csv', 'cenarios_utilizados.csv', 'compatibilidade_crop_marte.csv', 'crop_audit.csv', 'crop_mars_compatibility.csv', 'mars_pds_original', 'mars_sand_regolith_separate.csv', 'mars_soil_audit.csv', 'mars_soil_bulk_composition.csv', 'marte_areia_regolito_separados.csv', 'marte_solo_auditoria.csv', 'marte_solo_composicao_total.csv', 'matriz_confusao_terrestre.csv', 'perturbation_sensitivity.csv', 'provenance.json', 'proveniencia.json', 'ranking_similaridade.csv', 'scenarios_used.csv', 'sensibilidade_ablacao.csv', 'sensibilidade_perturbacao.csv', 'similarity_ranking.csv', 'terrestrial_confusion_matrix.csv']


## Next scientific step
Add real experiments with terrestrial controls: crop/cultivar, treatment, simulant, experimental unit, replication, germination, biomass, water use, electrical conductivity, perchlorates, and chamber conditions. Split training/test data by experiment. Test whether similarity predicts measured outcomes before describing this as a Mars crop recommender. Crops absent from the CSV cannot be selected by this pipeline.

## Sources and provenance
- Terrestrial dataset: https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset — verify description, license, and origin. Crop labels are not evidence from Mars experiments.
- Mars data: Berger (2020), Overview of MSL APXS Targets from Sol 0 to 2301, https://doi.org/10.17189/1519050
- NASA, controlled-environment crops: https://www.nasa.gov/science-research/nasa-plant-researchers-explore-question-of-deep-space-food-crops/
- NASA, perchlorates: https://www.nasa.gov/general/detoxifying-mars/
- NASA, greenhouses: https://www.nasa.gov/science-research/lunar-martian-greenhouses-designed-to-mimic-those-on-earth/

Hypothetical scenario outputs must never be described as measured Mars observations. Original embedded scientific data and metadata remain unmodified.